# NuCLS E2 · Tres modelos nuevos · V1

ConvNeXt-Tiny · InceptionV3 · Sequencer2D-S.

**Cuaderno separado para utilizar después de terminar o detener de forma controlada la búsqueda actual. No abrir una segunda ejecución simultánea.** La búsqueda actual y el Entregable1 no se modifican. Sin evaluación de reserva. Los siete modelos comunes se conservan en su carpeta anterior; aquí solo se entrenan tres nuevos.

Este bloque mantiene resolución128, tres clases, inicialización desde cero, batch32, AdamW, float32, máximo25 épocas, paciencia5 y mínimo5 épocas. Búsqueda: cuatro configuraciones por modelo (12 corridas, semilla7001). Piloto opcional: cinco semillas6101–6105 por modelo (15 corridas). **No constituye el benchmark confirmatorio ni justifica el R final.**

Protección: el entrenamiento exige escribir una frase después del perfil. Dejar el cuaderno ejecutando todo no inicia silenciosamente la búsqueda o el piloto. Guarda una copia en Drive y selecciona GPU antes de empezar. No borres bloqueos ni resultados si aparece un aviso.


## 1 · Preparación separada y conexión a Drive

In [ ]:
import os,sys,json,base64,hashlib,zipfile,io,subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
PREVIO=Path('/content/drive/MyDrive/NuCLS_E2_ADICIONALES_EXPLORATORIO')
assert not (PREVIO/'EJECUCION_ACTIVA.json').exists(), 'La búsqueda anterior conserva una marca activa. No iniciar otra ejecución ni borrar ese archivo sin verificar el proceso.'
BASES=Path('/content/drive/MyDrive/NuCLS_E2_CONTINUACION_BASES_22')
INDICE=BASES/'indice_validacion.csv'
if not INDICE.is_file(): INDICE=BASES/'final/indice_validacion.csv'
assert INDICE.is_file(), 'Conecta la cuenta que contiene las 66 corridas base.'
CODIGO=Path('/content/nucls_e2_tres_nuevos_v1_codigo')
DATOS=Path('/content/nucls_e2_tres_nuevos_v1_datos')
SALIDA=Path('/content/drive/MyDrive/NuCLS_E2_TRES_NUEVOS_V1')
assert not (SALIDA/'EJECUCION_ACTIVA.json').exists(), 'Este bloque tiene una marca activa; verificar antes de reanudar.'
raw=base64.b64decode('UEsDBBQAAAAIABN7a102uKvtJQEAAJ8BAAAZAAAAbW9kZWxvcy9hZGljaW9uYWxlc19lMi5weU2QwUrDQBCG73mKIZc0EEKSGquBnmo9aQUFr2GbTGRhdzbObIpFfBifxRdzm4B4nfn+f/5/4jjeOTsatEjewVEfjXZv/PM96M7BvmrAMwpY16NxkoFoghHFCeCHR6bLLChZ9aqsbvI4jiNtR8cevLY2eny62z+8wBY+I0g6RycKstZrOicNrJLd4ZBkSVUVZV6s66s6SbPAaepw9NpRe1r/w8q6rPKiqK83Cyb4PmFAuepbmbln7CbmEAcX1zovyttNdcG/oh4HOE7a9CtSFjNqO6NEULbrtIlAD3AZAzkPoeKSu2GlBeFVmQn3zI5naRrN3XKjzsiSC/p2mAT7VnlPq3tl5IIw+olpITtG5bGdn7hcH8M6mBP221mQ0WT/Av1FS6NfUEsDBBQAAAAIABN7a10FLDPFMgMAAD8HAAAYAAAAbW9kZWxvcy9jYW5kaWRhdG9zX3YyLnB5rVTdrptGEL7nKUa+ASSCDI4d6xxRqTlNqyP5WGmd5CaK0BwYjreFXbK7OJy8TR6hz5AX6yz+CbGTXLRFsoGdnZlvvu9jJ5PJDcpSlGiVAavxI97XZGJYKxBSFAKBpNUksRH8oEAKaFQpKlEg1JzCL+Tu/A+tqJVVsbdCuFmvoVDSkN7xPgTU7zthqbCdRjCqsg32vCe/XafPOBeeo9lyKwtYxxCk02QZeo+An/9GDrZK1UI+AJbYWrRip8AICS3qz58asprb8zM3KYjjGnaMFCUakG4z35N02fMv9iaTiVdp1YBVutiCaFqlLUg5WtwJI5Q8hobxjHez+nmzebHJ3vq2a5TOpZJ3wiorCj/y68em3ari0RK/cOTV7crFf88NQ2uw9t95XlGjMbB2eOg3LcqXPFIgZXynyq6m8MoDvhjdbNrPpvDkJ5j38yu4r9X7jgws+gWUwvzZsQLm2sXcFlHynKxcGbuxXIGSKqiU/oC6DAzVVQT9obS7RAV9zDS39PZJevUOsgyC2TSC2XS0yV2aWCbJtMRVJwvLdLAquHvInRBpGfQR/EVaUp0b8ZGyRQQ8KaPJFuEPm80jmH+7VX9a1CgMwRusO3qhtdKBvyEg05JGCQ22rOaeo8D5kh3hxA1BDaSc1tJnYeyHR9o3J79dUO4Yy3N2us3zPWUyH5LIZLNoMFw+uIiyV7qjEXrTMaYgjE/ZX2Z3deKKkEfjMtxyQywja4WD5jdK7pjFWTRbRE/DiFf+oNXrYHh6jrbYrpVu3IbFsHSH/cs97+nwfkzn5GU0+0H+0+VF/hnGYT7S2ZkvQ6feeHbgb4CcH24Hy9nH82EHyvhQ4FoX4/5ao7Uk9/hWQhLqIEmn02iepN9Dn5TBMfqLVq3qbBDPxwU4+p/zT0ofaLn4evjjOfjzbMhgzF7wldxBH3I5z9W670RdBnxy0thUBwsxwy6SZf7hNCykzO+lf+w49uwJ51kmi2FIkk3SxP9izCbbH1rxKBx8IPGwtSZbK0nhdXMm2IGU8XIsZH6cacTU9QFecwaFd3Gn+fRbOI6xCxBV8VXzqvhXTZPl95smy/+p6cWx5CCE3j9QSwMEFAAAAAgAE3trXYyZILIlFAAAljMAABIAAABtb2RlbG9zL2NvcmVfdjIucHmNWutu20qS/p+n4NHiLMmYpkVZciw5NOBJnJkAyTlBciazu4ZA0GRL4glvw6ZkK4aBeYfdF5if83seIW8yT7JfdTfJpiRnYiAR2Zfq6uq6fFXNwWDwvqiLyoiZsQg54/QQMx5WVZGmhbEZucanAg8Vi5JbZrC8rlgeZgkeCmNrbMI0icMo+fbP3B0MBs+SrCyq2girZRlWnDlGxDeOsQr5Kk1uHeN3XuSOkYX1yjEK7hhlGtaLosocowrzuMAv36K5TjJM/ZqUiyRlzxZVkRkl5oCEoRb4gFfZEYFPFtVJkfOm81WxzmtWNczk66zcGiE38rJpwo6jVe9lk3CQaJo+vH0nqYu+hm6ea43uuk5S7sZhHTb9r/HMWe2Ih3dFGIMHMQHkmjFvs3CpthRhyxBeXfBgM2r6b9dJGjvGq3dXnz5df3r27Pq/Ply/+u36teEb5tnoNIwXp2eT0fDFxAvjODodjdl4HMXe7fk0nk7Y+MX5eMQmp1EcjqfjqTeOhqPxeXh+y87P4rH57NmzmC0MHm5YQEdhleIkaA/27JmBvzorsRI1u3dJvQr4erFI7sU4Vz4bR4bpYphpNxPcuyqpWVCz+9oiqm4MgXOLqDpGksdQFX/kQHX4umJByKMk8d+EKWlHCB27C/Iwlw12R7Ni0I2IiZVtyTZUKRBLKbar4o4rtpOFkRe1aJlBVet1lYt22oPcTVGy3DLvTMfI2V2a5Mw3TZuUYiEpiNHYOhZxXydR/RdaqLIWjrFIWBpD5Rn304TXFi1yM5zb9oVxJ7e+YnTUVtdAQ8Q4xTmPiopxawuFVwxHGdbKS/crxnLLOnWMUxsHUW9L5id5LeWAfhyyG9ZWBMuw6JUMM9xaWwzWXksb754SXikpx0m4xDwwxdelUC1sLsMhZpaHxrJiMbbJ4q55KAmUZOpkDA2dTRIzqy6dbg6Md137LfvE+t2KVczvqF4aihyo4ZT3aCmmnqTUMN3SWXg9GqPnLZ/P5RJOx/lR0/I0m72BOrekOwZtwsrCqCqChecv0gJnsPDcjIW5RbIOo2hdhdFWddWlEKB9koX3lpKmOBHbaZWr+bsN0zCPWBzs0JCcdEtERb5YE48+CNaFUL0D5KKUvLZ/IzgWL36kSUIRb99vkjloy7V6C4uOPeq9P00Skgwv6IiEwlrqvKjDNuDSjcSJYPyw+nXGqhBGq1yaPVdGUcFqAnISsJSiVnZRhWSEouXEzMI8gfXxunBhlqbtiim32xq2ZDd2r2KLy1fhaHJmgYDtrth9nCwZScz4yTcaH9pZehUmnBmfw3TNrhHpKsu8To1uOTgTyD/JI2gaHYTBUuEk4dyNcB0ndRgXFzQIRJI8iZKwHxhd5RzJBWA74uwax/JROgvw6cYsKqDKtsvLNKnJKWFfygcuq2Jd0uQHPruphEQrkqcgiW1XNyYibI21i9yc+z43EE2oFfyDO2oysYF0jQhjzncOlqhxomaZNUSRm44pAzmFUdN+bGR7k8JpSk5u+Nzu5sm2OQn3xhtPTqfOaDiezr8n4FcFYnIBfAFhQGEKgTRS/DQC5Y3UaJUvbCv5KwEuIFIGx23WnNMPTjook3uWMm7a3ZLg96G6wcR5Jy3Fu9rl/NH4z6fHaBKYP856EtvfDTBRWDYwSJy90R4HQwwyj7CI3ZPjgzgytZv99fn88QkBn586I+/HZEtcyBU0QetyBbiqtFVdBKsMStcRl6Zb9U23quy+QKQCCneDrVDoxXhl4LNDDNJIgxS8BEVopN0jV7Y2D6ohwFWygQaTvfMi3TDL3l2cVizdhAcVA4JMAGfqQhDoptiH+Pi4BlpbrMEDZNXa9A43AjQInCZRQylwQpLN9twjOEkyN4MR/+SbH//4B9Mg6cH/J1/RYnmjcwf/DnLyBqgX0A8yATaAC64LUhpeVxTM9xaS3gxS2vF2GgZIMhthQjnHngvcFZ4iBtWCtHVjmh9iVAgiN7BJOMcIZ/cUnySMn0fD4dD3hzNEIIoK5mdWiWkA96bDncQxT0yHrAEq5SzSNV/5v1Vr1gu+UjWfPSP14sarqgCYVNBaqSGFjyCA562DwOIsXRAYLGoJCRGd18uMQKfElDODRrhigHrCIPHUDBR9olm1aKuAWbWI3UBLgzbQkrK1wUtWk0W1XCWa3VR+OwVRsgOdO9rWMtu3BaWCxr2/c+ZRUW61M8Yp6Fvr66yYXBX1dGjdNymXSz/4t2TWeP9EtTEFMM1Ld0Is3N/MnNns2Ot2UfsyKaLEJhAJl3WvWLNd2Hy2hhsZOUPHs10JIuyT0WTSzifYo0jUSBSgeDdD9+x0Ohqejl+MzkYT7/x87Azd8WTqvRgi4Eyno/Ppi/MztJ2NXrwYT7zp2elk5I3PpnMb7P0CPyz+63jk8d4KI+90+mJ8OvEm51NvOjwHtRFoTM/OpqPR6XA4BVG0eS/GWO309MX0bIrUa/TUCko9rPqY9mOf8C6TcykRurc6r9mkBoygYJpa9NCCICF0apHNFwR8D7TK7QC3rMM06DqasCO7o3UckqsMN2GShrcpfONM79Kmd4wo5rZ5tKqKHO7MitkmiViXb8l3V2QsvkmUzB5Zfa7kiKV705AntrPwvDNJ8MAQl9cUhMjNAtKnIq92jB4/os+loZbIbYpbQOL5hbGVP4AZ3B+6EKM/7PJCuWxeBEuEpN0YeO8IACJX61tRWiyTmvtiTWg5Io/iBUsDVee5u1jnoiIRpi5SCM4DQghkDHKqs9UnHYptkrWEL8jHMStND8aQD9/+UVEFgeK+VoghYCom7oVZkouLNJ3lsWLF5cWipqwFdhmVa8tuUw37YtuO3XatO5Lg/EglBeDxOfnFrX2RH/nySQwOdY8lWGgVlLaal90+Q9slDbQpjKo+vEdYBn0qdXU8B3Ez9T12PDksFSwR3iYkjpgRmNp8+zu9cLMXZYj3k7yXRm85pXZS7aTnZRYyyKY0ElZw7z4aXEA9nOgJdbpUGTgyg4/Xn/787rer179+cr8mpUnGmZW+nNMrpejFE6GGqszl/k9SvsGvhW6HChVt+9sPwevrN++ukMGIMPC1r6kl6SmnRCwWDFXLtLi1zOemvQ/aBGYiqpAx5QtlU9QRWNul6g1yAZFt4WdVZyn91ve1ae+jn6+uqsY4VK7pcBiJbL+QI0XROwLZpFzNOsvCaqvJG0huncLObkRNiQyRW6VMAUWpSeWZu9tXuz+R05FXyF3Z9lydIjwgnEKLiOn85PaVKUR5HtySGGKECZaz2ht5eAM9PE+GuiSqyu/lZpJjhY6FfxCJGC0x1yO0qFVVM0otAdHXrO0jX4dsnnRSaORN1eQlMs8DVVWU0DOIqprbepSTBilpubyOrTguFr4HgWFtskzZZV968MmwHgpiXSQnAblhWZLhyzqI2IjYhcN+Z9FaJjl+A+MaloKMxUnYX70pZ3ynstBOB+cImNEG/58cIiKwTWwkXMiPmBY63Bt0Oey2pJSwKxtCPU7oHJHW5IHcFZe6LjYth98W8dY3X668yzeUs/TL4b+sX737ZGxGL0/Q/7K8/Kx5XaoHJ18BkCkTDMEGWyaV8e0fZRGFrvGxUUeZ94ai2k7VUYGpkWvBAkTNJ1lSVYgoXtAw3a8jgC7FXEX05Ul5abZMH4Hr8vIdHARnWZICZRhZAh0GiSppHCIt3itVyDJGkWuJIyEVOn35qq2yq+idIawQHIpq62umKqRtaXYAJykTB8o6JIdAQfaJKScnoTLUnom3K5QF8gnumwa8FB6thfkwHh4lzyfnQ1F1I2VUXNjHHmDmzPUWj87DyBsee+fD56ubAUQZNMo2mIt+U+W7q36+2xDqlpcCXkAvRpdI5AdyU4P5o/Gvv/1fI3CqLQzUM7qgJCNSkvfsdywiVj5+483EqM6oB3NQ07gaLx4v1AGLkRnNDhi9g6ZrvFd0IJFGuwxr+K+//S+23B1Vj2e+WRqbhN39obj3B0NjaJydDY3ReDhAAIrrlT/A+wCchohGsT94P0bv8DMk96ezUwxCvEj9QQ6LGhg4v+IL8weATNvByeVLOiYkBIPJwNj6A4y+9F6eUONuF6gNLodNX1mkW6p3Ncc6eJAPj4dXq1mYNm/HiudTWv4EW1M7ltb99pc3v358fy1Dl63fTJgvf4qLiHCnQZ2XLzNWh0a0omuq2h+s68XxOYTA623KLkl0Dwt45+MF7CTdzq6gn+kFFE0uP5sOh+U93qtlks/GeEbSCWOiPR2vWLJc1TPPPXsEew/dJG84/BlaIZcwj2iRXjzUMYeqka7zAJKwhO8VuYKj0mOH7tj4DgzGcJ+aXS7U68RS4ETanQD29gUGudmXOKksdg89D4ovWgoOF2thgHSTevh0xeBeqUil+P8NKRZZmTIaPTMdwatgVcvvL7RLmX6mcSFMyRf3XoJfWwPHYnxR1kmGjKBS6Zt4d6/iMPuLjE2ExzCxZhX4c9JKyiCtnDtxEkHMonArG/UWVWhlTeL5RwR7mD8wpP1ETlVXfne1Z8nKhCArbvGcnWqjI3bu3IZ1BPSHDUgWxLvDVwBeQCI0xlk2K/t4cuCGgrui+oLt+OpaYhP+0MJ6CfOphQ9Qb7w3UqVbxmv/2HPFg/A6K394cRvG+J8DrzVtngytKxZ9EZbrC6WBxiRZ4Za12ShTN+KAAv2HvFruxnBKUcpExkgqZBtRWJWM7hFzpFShshDZj+OO6F7HbelFX9RJUgSyOrJAKGWQwlGSZJCrlmtTqQaHdaVbVSjqihFCqYhIAB7gQOTlyhcVyyDbi1YnDw5re00NmEnW6O5VjBECVvQcIT3h6Ul4tiYhUnPCBZxVm1BhBDK0iiDDqw9/pus9BPqCij1CLrdoJ9FVxu9rKk6HKeGRb3+HfVRFJytRDCauKfJ1uxFNroRU1g7m79Xl2X0ZhJslgLF6Cvhf8QJnF2gNB7IGKIUiItaayR9Rk/e7x10foKspSU89m0pjG4Gqd6WkrZjFK/VBj0VjGONN12dqVaOOPK3SrkouteBMnFu5rVdFHlT5kvRAr8vUQTdKlMCaQerkMQAN2iDRrga1a8ID7NCSNYjdcU+UYPQKjL6gcLhETU7apfZUbUYrzeyxf5iSigofAcfBhKj7dhHBlIDFdDTZ75WBSc9Ej4CaojKpDxe+TOAhfuT1739wrpfS10k0SxdzlCKIiZeiI0tyCab4zLgF0vzSJU77xa6LOol8+hiFSpiLIJLflVgqZrnCz+NN/AZNkSmHq1Q1fVVt6uyNlY5172ztPtysK8fbsZTOIunuWlao6ATqIiBopILqdypRqlqxPWRI9Ce4/YFSlZ62a5I+VKTCbHv2/SpV//Ohp+pURAnRKvpyF1Yx5JsXVUa8yi9uojQphUACag8OQYCJ6zBaPEgWJC7JoKZh+1Kmo+mdZFvWIl5ihKBohXTkcIWr+VOnfiQuxJsCG+Qn62u+qgbZ8vOAPZESCz978vpEGtEBu5EmAJdAN6JCn5oLlfrAhQr9bVJn61DZzd+ppG5Cp1FzCM5vv1GhsR3X9sUBu+hshkX8kIEcw3A6L1rc+SI0Crvz5RY0m+keT3LV3n4boUSKDsrexHBsaL+aoOd2PrajF0rE1Jai6Gze0MnZcg0vxX3aywHK8gOLQH7Zsgpl2QNuf50nf10za0dathAfnGTcq4aIUVSJGdpNNWMvrjU1F4ir5+V3dnNJEa3vLEQI3Bmmx0HxvwRxvXlUodSQB9UlLzSUIg1Lgza2gyEwEq2q9xRwoSLMDCse+d5OZEDiTnr92CTN/gNp+KNKZf0Hweyj9E8PumqI1Fgc9BvPf6Dt7ubNRp2wrCyIYsRFfs/NgzZBBbriTl3FaGdLxTpPGZ959fntp19n6qMnZqxzwK8CSb6YdEGfflRsiaOrVMGnYvQN4bd/Uk1ZfBlSRJRBuYdZEGL1u/qaf0DaXeKzA9CaAdrxOg06Ur8HK24d5tSBcv8M/y1YllnNU+R9+s/RlK97dEgD8c/pIJSvwNOyQVi20yInv8NWyxYPHV64BVJqW8seVkFK2CIoSrD+LbkO2PganFruwSn7CRAmS5AddNphZ38WQS2tbtmlDI2BqsRq10IFwX2j7DIfTd/abz4FwX7NrVEe+6Irloph8JKbUNVImzEtyaZcoVUcbO02bjcx+kGNk5I5oHIqE/6BcEYm6/cdsyqzIJzEzcdyTcHT732X46gU09dv3x1h8wHgYuqr+9wb8hFb+txtfkiH5ITGsfQmUaOcVwb1OkPS38WIm8QZInOHOJJ8UURJXfQ7PdHJqBKWhf2u0Xz3u5GDn/IcLBXMdwrl4nwk87LWH3T1SqkMSpTqi1HQgfQC+rRBv1pR7doFzVPfQClyjzqcv/dvkic+R1KLt9/BNQR9Xy05v8haQHOT3EPUQuj3ep6iMd277mj1Qo1wchHyk3vtpgMolgfq08ushzU6nNFDGSrSkAapmxOgecmbbe9dVCj7qKsEeUpQAsg2TEnp67zLuTykCpyfU6VTfLi8CW37Zqjn0RcUQvKluBHX78t/8EI86LKxiT2TKYdcVXMy34GKOzRGw52M57upVrPSITDa7Ks5ROsJPGo/B7gengjZS2qNV6BKp7931dXgEwHAtSq8HtFkJhmIW7E6jBU6bDzlvmNQWUpFcBNJQOmSTlMgaW8y97MZW4OomCNuT+S7bkFPr9mZro7xWWz3IPUPV00deCZWidw6yKRyCnuEmxeIN2/8EsK3bLDUAWEn2lxRjfS103Dkl6u++ebq03Xw6tf3H95d/3b1+ir4fPXu7eurV29//cVslL0Xy3ZK1Y58pzAgcNw1fS9OS86QMalbZj1k7cEz+jJQxpQWdiEtgbQOf/qyUycBAq23QYR88TvfvezVavqz/h9QSwMEFAAAAAgAE3trXSFp30s7AQAA4wEAABgAAABwcm90b2NvbG8vc2VsZWNjaW9uLmpzb25tkM9OwzAMxu88RdTztGpDXOA0tlzQaKfuzwWhKGsMitTGIU4qKsSLceXFcFsQHDhZ/vzL5y9+uxAi6yCQRZddi+xQyb0qjvJU7tVpkc2GcYsGGiQeP3DLQo2uc/AaVbSuHxkWravBR7ZR3eWPRvCSgPWwNIoy1h5HQ9KNNXpYl7NVBBdzE2wH+X2/GWuR1tu9kkv1X5w6aQPB4fD+FwxAqkjQITE4t7535wkHitqM8K6Su1W12pSqKJW8k+vjYWhkodbldnU74ewDoRvDMbWVspp0DPYZnGoxYhiTe06qI+Sx9blLdUMKlmq4ysQ7jKPJYilqDIH/S8KAOH9+EN/EaNGLxdWgeNuw5813FehrvqFuBIHXgZPPhEO2cE82tJqXW5yLAgUlijam/q+lD9hZmp7rOibdzLOL9y9QSwMEFAAAAAgAE3trXRufjbPkDwAASyQAABkAAABzY3JpcHRzL2FkaWNpb25hbGVzX2UyLnB5nVpLc9w4kr7Xr4DdB5JumnrY8k5XBSdCLZV7tStbGj08u6tQMFAkSgWLrwZIWSWHf8we5zjRt7nqj+2XAMhiyeWxZw+SSDwTmR8yv0zq+fPnv+bV761g4r7OK8WbSsmKTXcjdirUXObs5Z/Za5ZW5VzetIqnsiqF3iqqTOQV9dUyr5qKZYLtMS0KmedcR6P3FeMzJZgSWqg7HjE0YI6cy5Sz6Y55r6mzbDgTebdKWhUVm4kyXRRc3dpdVWFlikbPnz8fyaKuVMO4uqm50iJccL3I5Sz8qKsyrHRY57yZV6oI9VKHjSxE+CBrHEOM5qoqWM0bGs7cMqd4HVU6EuWdVFUZadFkYs7bvPG9g8tfj/fPk7+enP3n+en+wTQ5OHn/9ug3L/TGr7d/eTP+kxeMsElES0ayxDkbfzvUjfJpVT9JaNckCSIcs8rvhB9gqMKB9dXO9ZZnNai9IBj9dnZ0GF999nLljXMVep+EvFk0SSZSvvTGn7IvDCdiuWKyZP6OePk6xK9XgWn9lJnW7YjaXgfXo/Pp9PA8zqVufMXLG+G/2dneCfHrDXbC4Ri0WcjUJ7HDjDc8GI9YU9SxOccn2SwS3c7n8t6MiOzzz16EIV4wwe/ok5KNSBpx3/ik9Shri1r7tFQoywwHjHdDUepWiYTrVMr4Lc9hKZ7n1aek5KV9D+xiSsBiqTCbWflgpwdRrssn58ZykbjHubRPTdRmts8rnmkrrBI8s4IFwbOYJo8Vl1qwDzxvxVSpSsGuKyg//lGyDCtKoHDMvJ/JeEaQYARQajH+SldGwkzeCKi3DsZKNK0qmQNhpBd8d++NX1tBZstGQNYgWoh7N8XOr3l666uqaozqAWXRxPTq8LFl+qKSF+JnLzmbnl8eX+wfnpxHQLI1QWwnrVnLWWjEqJE50Ef/I+u3+OujD7jywr796DQ5nL493r+YHgaMa/ZAKiVA1YQnjcshMiuHusmrme+9AFJpjLFFJLWBt4+5ZYZ3K4SBohel+g63JCLr0N/mvqE/i6bIvW48He5Z7E3/Y3pweXB08j7ZP7g4+rBv54wfLMb8OiRV4kbLOwCusjoLDFx74FhVBBNnCvs6Mnq+47mE1UQCt1LnAguqtgxp61ALkdFx6hhNWx6uKO48zyorgAFcWTUQtEecW99gd8RUPMTeGvDMZF9duRvuXYd4dp7RvgAJ2Mq7Bkb9XprQe7t/Tk7m3enxFPbeTz7sHx8d7pN2vGADjs86maF1OEoNOQEe0eEY5yJZyKRza5dCfKxUVJMxMFEWlX1eYCL8K+/slbZw2NoZET46k6l1+onVJ7243kI0Cg5dJ3CmCVAtSQDT191QUiKJsjUPBpiBTMMOqAOKw59Eywex4ahQOoWJOwnvkko+OCKm45CLoTVM85NTBV9bKIePWcACZI+6okOIjyJtoVCuvWsSkc+0X/B7//7Kw8mTgqeqSuY76CSlGrAvgpfK9Dq9eNcwez8u+DN88i8bzvPvTrh1y9HldSi7UK2wGC64LH2L1C7mRfvqpi0w45TelB9M6ohnWcJds++9fAnYI7aEzbIWMYWjUInfWwljxrT0phnaHOLbUyDBV3PmXAsvTBeVTIWOr7zaMAYAY9ZqUApazrORHcB3gTXuRm1ckbekmAeuvNV4GsnjOjKHp9G4jsaEPCIBnvUrGt/Co36NzsFc7J8lF/CjyfvL6YeT8+TDjrfBKNOygfflBYEYJMSQItyuMWxyJzWHZ7R8aMn6DYgzwXXwwuDe66Syqux83Pv24Pg8me7+gAiXmrOcs5SrWhDg79O81fKOs2+t8WRH3DC9LHJZ3vqbPMa5GcVEmfMHwBx3kM5UyIYMT0p268T9gj11mfDIQCp2f1c9axLEXb+5P66RLko3zTEg221Hml63n+veIPshjd1yC6IPu/Oal0Oxo+I2kxTAzRoGtaFx30l122OYjFnFTzbcWhlp//CIPO7+MfQ8/a/T45Oz/YuTs6MTb+sbwcqSE7PsIFgY+c/akhioO8ExZ7PHf9h7AaQ2ApxWsQZwE6wtOW66AjfmKYKd5ctYQZYS/k49/i+rGgXLkYeytKWU1A93IZXp162DbGVw+BMtgPBYIx6ib+YgpXvja0acvyZhSkZOXskbCUDrFJEX5Egzw9Otdculf78GrpUHXEFlQBO+CT14OyMB3AXTspg9/pHLlEjwiOVVetvb5dvKbtSSQoshOTQlqmo4cu/eMyxmPu4Zqf8ZrifzxgAdCAEe/QCxrtKNN+5yhKhEhLbNintjslVEv6hpBqqR0HxD5r2tWlXpFuj+1q1Qpci3wK2zqtjqhq0FGEQyJWtoCbr70ekddAzvhPFK8SWcEw29JyMyInFTM8QodCPA/puDiC6BpR4JBIKcl6Rtg5+0qiWfsDtAzyRiWpK7GaCqh6UGQxYdGMk8pHdxTwFS+Jx4hSxB6JdjY4O2NLAopNayvFldNxPBVrOIyNu8q4SBlmSwsg7hS9OF/Q0/iyBKaVvRD00r5BF3uzSYHkfM5HEcrARDeQ5aIna7se9ODqfH5+GslXnmBpp1o7aRuSZ/w7uh8Cf8GHxBqJ7s2aFpax0pv+My5zPiKxu1PUXQZmlLK8CPmlgpkPD+dno5Icdqby7APgwqomQHp5ddoKBjRkkCa9CpkwSxYifajnZ3vY0bnovVLjQ1jt1oso49JcJjBpoLl15SVpMmPL9BpGoWhfZt3LcDZ0hARJlpOmxZRn3CHTt2u3HU2srGwBtWgxPmTdHmkc32mvmrXbvo5p2fjhqxDI40FbEdbV98jxb2DOhy4hnr+XVoXgkb0ZO2H0zBt1JcRmmCEYAW1UvvX5u+jkWaf02BBqwnrfIYnY3fyLqKPcvt303fX5zYQG7werIeaJL3J7bScPbORZ7QbWSTeovxIOxqLYklWLHJ+8MbRY0xVRTCGW/SRfwKybjht/HuXmg5OoAZ74UwZeJ60IFIAtCVYHs2W0dDam5j7M3Bq5tXu15o1NDSWeOd3T+FBS/lHLlMbHSPM0wPkE2GlicQP3cRPsABYM/489xQorHLhefBKjkxhv0SumoRONNJcjydnnk4gxbwLSKmFK0Uzd42EUNkA3G0vRuMumLBKm50egcDlx/7hC7smsmjlnfOKIQxh7TBNRy6onjwvD6GLuDTCxwatxaX9VpjvWwWWKkPOva9G4BYc1O38cD3IFolFvgJqcvfDjadk/wJHM/6KXE0jL1RVVtraxUTlcjv9cYYGTcaGR8ZO0/5E5veSWIKlCgrWYAINBUSS5GzLo/KEQHIBRNvfPwbwWaC+IuGhUhv60oSqeOP/yiJldy0XIFWRG4r8JoFEvg458UMs6u2GePHdeq2gOdZftVJkQOJnD+vcnjYVdIeLurAliqo/fuUb5IRQu3oLd/VVGz6aGoAgc1Tvy4V0LxhrQB6oRTl5N3RBbJy1uXp3kC0Odj6wm3rErlhXkorflVqeJqZUmVTr1K991hc11TncATb6WONjP9LN/3FC2gQ+wwQR9euSWrBb5NCFJVaJpSRg4xMLIbaMgEfGZQqLMJCEjW0UDWSa6qBfl+VNNTW1axGzJ59/eEzEuj7hAJDikVcFQ3kbCUu9Ts5+2FE2ajdepDvTetG+cEXksaZ9ow0nSPj80JToesuWzA0LIb/f00LH0R+bkBybUY5yCNsRhv3Ge3YbEerIuxdT+giw1T2ah+oqtbdtXYmufKMF0e2bWusmP4Tu0RyQekSaxDnONbjecjA73CnQUjooutJl9yCuXDiR0hHHv9mizsMSKSUKxemDgQv4eIR7je7D+Hv6IAS5MBfkSrfiWqhaeo6hE+9aOdzqMBVgIPJfXwfNZXvQDRZxsvBq6tIEnIoSNjIZ2qQ1nUATIQA/9+2t7eDiZHKejQD1WC4cFU3zsfiSRbRfsaLv/pmCsV0DIf8QHyYK0SW7R3a+8euCDE0YxxoRJsnmkpySxLa1uB3dm3tdLikXoJ1qaqEaswyaWwSEDIErk5LTNx3p4qMVX1zjOhBqCq5UTzzSaSmSkpczh6cDF75RgIuZqJ/Hxi53NnBuOZtCVJPbCVKFXoSYqhVvfTttHBpF6ECmWmgmnYtnsU+rPeqr9zZ5aRGFkB1Wtpic9ZXAucZm8NKeKC6nbGMk1Nrwwc/IVr4HTWFiJasp7mszTFxPlUkm2y1FwraJ5Fz0oGVxblfUhQIeu0H31M7rBfxGklk5m/Q/0vYJXCm7YYZPmTOTJyYpwvcbrONFmlsO0EBCuTUvDRr6qvd8XVAMYFusiUf9g7FxivOkR/k8NQW4FfUdn21fR26wyqiU23h1xH4BWwarOr0XyuFiOFNi+RSJ6CFIjHIiSFZCBclC1OlTPohJiokmuhSP/DF6z2EDOuTQS7TyvrS+PsemFSS2MQnj63O6Bim2eSLXePLnesQMALR29mFe9GVHhvqwRs5k4SUzORQmShaSE3ep5rBbX1sKefKqFzC6HBUBEHgz6RLr8weJrvqvWZnM/saTLoPOtYVh25UMLEhYPA5y0144vqxeW6VHgJhk4FKRFE3yyQFGkxNjFltA49kOVvz/672XWXZfOfrJFsFyyHHRWdNqumIn4EUVTH0YDl43WRnu4v8nUAvdrZf/LL16s32dvjVBNh1d281cnevG2mN9ZpZE2j28+BjL0O8cCEhYvuPf0c6pFhXEn/8owzZ0dYJWzLwX0NITFVClmneLsXgW68xXshUNaMtHsi2yL7tEpEXDOL06fTs7dFxx8BO/nm03hRUbQBffeSxYFiVYOhjK8j28AuPGzIM6MGzmEa5fGxzjcB+USDBC3jtLsgCwUSdG5ixj6WEWhNjyTkPKEIfd5OFxPSVS9jEGvqyOwWbriz/+cu342hDzH4tZoFf0xhBrkbBYj6lki54KfpYNsBhv93W3LP/IJB8ll8cI0ZQ3iGyPrF79Bfxn3wuoU2QipAfEGOnND781wMqUYGygMrYPJf5OaAnECCAr0+ZfQwi85Uypk83NvqSmawUcI+3ok82Po5t69XHa1Bdqy7rfa2PXvunh1iCOpMyruR1iCOAgcAn2lVJwTZB603YfRUZ4PbXy/O/XE7XcocfRq79iDI06wCd/aZPcwksAY7gd6I8i+ltHbBr4dpijbRMH6YMezd1F1jl3N72Dsrkr2dg+y1VpX8H55IzxV1dm9xzSh+QCehw9+YGOwwSZyN8mWrFuEPdBmSyRR1/vh2vWeXq1jrIW/s9M1fek/+VCL5M1hHq9LaePm5yqfaT7NCb2pbYOz06Prk4SfZWX2aHFZmVAwhxZIW11hpf0LM5a9AVNxIB8LTQVVdooQBG349i7xCqLbXl2GemVl5X1mdOEA/mQplnUzmHaTScpahzUHTnZjvXCXJP5IVKOtyFU/po9Q1nao73Y84UcEpMTSJJgMkkoW+TSeKN7TfK0f8BUEsDBBQAAAAIABN7a10vmwiafAMAAJcGAAAZAAAAc2NyaXB0cy9wcmVwYXJhcl9kYXRvcy5weYVVwW7jNhC9+ytYAwUlVMvYjux1bPBQNCnQQ4tgsSiKLgJhIo5iYmlRJSnHyd/0A/a0n5Af65CykzpZtBcDJmfevOF7MxqPx1f74AAZ7mvTe72DLbYBGZ3p9mwHRisI2rYMmNK+tizgtrMOzJq1lpJSrkOPbgdiPB6PdLwODNxdB85jUftdsQG/Mfq28Js+aFM86q7RBkeNs1vWQYh37JB3TX9HV39cX/308epS8sXsHFRzvpjPJu/nU1CqPp+VWJa1mt4uL9TFHMv3y3KG8/NaQXlRXkzLejIrl7C8xeVClXyksGFb6iXLVyPWySMv8aO762Or1/Gfy/J1Jwi+gsNxxt+9I568CA8dysiqcPhXrx0q+dH1+K14n17rP1JAdiJVj2k+y0fMWRskiCFT0Dtas0MiE8/F9rPSLqMEwvcJosC99qGynwfAETO2BiNT9BB3xtu+Nr7CWUXCWV81fdRTxF5GTDekWhiyRMLy6V1Y2HZyOL3XYVP5vmn0PuOitp2Gim55vh7Ui0cPUb4MImhBd/mafoh8Z6DGLMEQtQjEDlKLP3X3c8wZLhl49hjLOriXj5QJKuNbaHWj0QcryDOcECLdg3WE38BsvsgoIRcb3Ct9R5FZ/p08emVFhvXIfgfT45Vz1mX812fE6N2gW4Jmv1nm8WDcllz39PcdtuhFKgjGOHvvpaHwjFiIS12HD0SPHEKlhcLaKtJH+M7oYDQlZnke5aKkT4411jHHdHsEih24T5ykCbqmKeI38TLjabp4wV/mi9OjtCoGE19Qlt9IyXVLMxktdRO5efIpaYdtFqFzKaeL+XJJN7GqLlJdbMmNDgKmmGKaxGXKB5lFk5wRPrh6o3dUIH/xWww6eINihfaVQ0O8dlgFmzLzb7zvhz4AI3852g3Ikt34EeoAk4wytPZKyRgQha9uH0J6xX/LKiURpcDqmeyqti0p2GPqBwIcfXPS0aH461IU/so1b+DfdvfLFsgZR+fAivEf3tSKTQxz9//DGodExoTTCRtmKw7QvdMBD8+RKJ+MFWUe29PfzyaTiZSTVed0XD3XFAQOav30tU1SoAdqwhi74oUu+Bkvnm1TNLTmN8f9wQZbvBm+EzJx6Gh3DrUuo8wsDoj1KzZdCPLgyxidVl8fvwzP34qnL7TmyCTURVW19K2pKvJ5VcUNXVV8NWzq0T9QSwECFAMUAAAACAATe2tdNrir7SUBAACfAQAAGQAAAAAAAAAAAAAAgAEAAAAAbW9kZWxvcy9hZGljaW9uYWxlc19lMi5weVBLAQIUAxQAAAAIABN7a10FLDPFMgMAAD8HAAAYAAAAAAAAAAAAAACAAVwBAABtb2RlbG9zL2NhbmRpZGF0b3NfdjIucHlQSwECFAMUAAAACAATe2tdjJkgsiUUAACWMwAAEgAAAAAAAAAAAAAAgAHEBAAAbW9kZWxvcy9jb3JlX3YyLnB5UEsBAhQDFAAAAAgAE3trXSFp30s7AQAA4wEAABgAAAAAAAAAAAAAAIABGRkAAHByb3RvY29sby9zZWxlY2Npb24uanNvblBLAQIUAxQAAAAIABN7a10bn42z5A8AAEskAAAZAAAAAAAAAAAAAACAAYoaAABzY3JpcHRzL2FkaWNpb25hbGVzX2UyLnB5UEsBAhQDFAAAAAgAE3trXS+bCJp8AwAAlwYAABkAAAAAAAAAAAAAAIABpSoAAHNjcmlwdHMvcHJlcGFyYXJfZGF0b3MucHlQSwUGAAAAAAYABgChAQAAWC4AAAAA')
assert hashlib.sha256(raw).hexdigest()=='37e916dcc735fe86eb2bfcc142b6ed1f98b1a468337fc07e03b4b9c6b291dd05', 'Paquete alterado'
CODIGO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 for name in z.namelist():
  target=(CODIGO/name).resolve()
  assert target.is_relative_to(CODIGO.resolve())
  target.parent.mkdir(parents=True,exist_ok=True)
  target.write_bytes(z.read(name))
print('Código separado verificado. Salida:',SALIDA)


## 2 · Verificar entorno y preparar únicamente train/validación
No cambia PyTorch. Si la versión difiere de las bases, se detiene para evitar mezclar entornos sin documentarlo. Los datos se extraen en una carpeta temporal propia.

In [ ]:
import csv, platform
import torch, torchvision, numpy as np
assert torch.cuda.is_available(), 'Selecciona GPU en Entorno de ejecución → Cambiar tipo.'
filas = list(csv.DictReader(INDICE.open()))
original = next(Path(r['ruta']).parents[1] for r in filas if 'NuCLS_V2_RESULTADOS_pool_corregido' in r['ruta'])
cfg = json.loads((original / 'configuracion.json').read_text())
actual = {'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__}
print('Entorno actual:', actual)
for k, v in actual.items():
    assert v == cfg[k], f'Cambió {k}: actual {v}; anterior {cfg[k]}. No mezclar sin documentar.'
assert platform.python_version_tuple()[:2] == tuple(cfg['python'].split()[0].split('.')[:2]), 'Cambió la versión mayor/menor de Python.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'timm==1.0.22'], check=True)
import timm
assert timm.__version__ == '1.0.22', 'Reinicia la sesión si habías importado otra versión de timm.'
print('GPU:', torch.cuda.get_device_name(0), '| timm:', timm.__version__)
if not (DATOS / 'manifiesto.csv').is_file():
    fuente = Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
    assert fuente.is_file(), 'No aparece NuCLS_V2_DATOS.zip en este Drive.'
    subprocess.run([sys.executable, '-u', str(CODIGO/'scripts/preparar_datos.py'), '--zip', str(fuente), '--salida', str(DATOS)], check=True)
print('Preparación lista. El perfil verificará el manifiesto y las imágenes.')


## 3 · Perfil de los tres modelos
12 pasos por modelo, batch32 y determinismo estricto. Deben pasar los tres. Los tiempos son estimaciones de entrenamiento; falta sumar validación y Drive. CPU fue probado localmente; la GPU se verifica aquí.

In [ ]:
import datetime, signal
COMANDO = [sys.executable, '-u', str(CODIGO/'scripts/adicionales_e2.py'), '--datos', str(DATOS), '--salida', str(SALIDA)]
def ejecutar(fase):
    SALIDA.mkdir(parents=True, exist_ok=True)
    log = SALIDA / ('log_' + fase + '_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.txt')
    with log.open('w') as archivo:
        proceso = subprocess.Popen(COMANDO + ['--fase', fase] + (['--autorizar','EJECUTAR_TRES_NUEVOS_V1'] if fase != 'perfil' else []), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for linea in proceso.stdout:
                print(linea, end='', flush=True); archivo.write(linea); archivo.flush()
            retorno = proceso.wait()
        except KeyboardInterrupt:
            proceso.send_signal(signal.SIGINT)
            try: proceso.wait(timeout=60)
            except subprocess.TimeoutExpired:
                proceso.terminate()
                proceso.wait(timeout=30)
            raise
    print('Log:', log)
    assert retorno == 0, 'La fase no terminó: comparte las últimas líneas. No borres resultados.'
ejecutar('perfil')
print((SALIDA/'presupuesto.json').read_text())


## 4 · Búsqueda de los tres modelos (larga, autorización obligatoria)
Solo empieza después del perfil y de revisar el presupuesto. En sesiones nuevas repetir1–3. Omite corridas completas verificadas y reanuda la última época guardada. La frase no queda preseleccionada ni guardada como respuesta en el cuaderno.

In [ ]:
respuesta=input('Tras revisar presupuesto, escribe EJECUTAR TRES NUEVOS para iniciar 12 corridas: ').strip()
if respuesta != 'EJECUTAR TRES NUEVOS':
    raise RuntimeError('Búsqueda no iniciada. No se modificaron resultados.')
ejecutar('busqueda')


## 5 · Piloto opcional, separado de la búsqueda
Ejecutar únicamente si hay tiempo para15 corridas. Cinco semillas sirven para dimensionar potencia; no garantizan significación ni generalización. La búsqueda debe estar completa.

In [ ]:
respuesta=input('Para iniciar las 15 corridas piloto, escribe EJECUTAR PILOTO TRES: ').strip()
if respuesta != 'EJECUTAR PILOTO TRES':
    raise RuntimeError('Piloto no iniciado. La búsqueda permanece guardada.')
ejecutar('piloto')


## 6 · Estado y evidencia
No evalúa reserva. El ZIP contiene registros y predicciones; los pesos permanecen en Drive. Descarga también este .ipynb con sus salidas.

In [ ]:
from pathlib import Path
import json
carpeta=Path('/content/drive/MyDrive/NuCLS_E2_TRES_NUEVOS_V1')
for nombre in ['protocolo_fijado.json','presupuesto.json','seleccion_hiperparametros.json','estado.json']:
    p=carpeta/nombre
    print(nombre, p.read_text() if p.is_file() else 'AÚN NO GENERADO')
archivo=carpeta.parent/(carpeta.name+'_RESULTADOS.zip')
print('ZIP:',archivo, '| existe:',archivo.is_file())
print('Este bloque no modifica las bases ni la búsqueda anterior.')
